In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import Galactic, ICRS
from astropy import units as u
from modules.vr_opt import VrOpt

import time

import random
import nbimporter
import visualisations
import consensus_clusterings

import sys

from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
import itertools
from tqdm import tqdm 

In [2]:
sys.path.append('/Users/lui/Documents/3_semestr/Project/code/local-clustering/SemiMetricClustering')
from SemiMetricDensity import SemiMetricDensity 

# Remove background

In [3]:
def compute_density(data, k=10, agg_func=np.mean):
    nn = NearestNeighbors(n_neighbors=k, n_jobs=-1).fit(data)
    dist, _ = nn.kneighbors(data)
    density = 1 / agg_func(dist, axis=1)
    return density

def remove_background_noise(df, labels, features, k=10, detected_labels=None, print_info = False):
 
    if print_info == True:
        print("Initial dataset label distribution:")
        label_counts_before = pd.Series(labels).value_counts()
        print(label_counts_before)
    
    all_idx = np.arange(df.shape[0])
    density = compute_density(df[features].values, k=k, agg_func=np.mean)
    gm = GaussianMixture(n_components=2, random_state=42) # two distributions, one for background and one for clusters
    gm.fit(density.reshape(-1, 1))
    gm_labels = gm.predict(density.reshape(-1, 1))
    cluster_idx = all_idx[gm_labels == np.argmax(gm.means_.flatten())]
    df_filtered = df.iloc[cluster_idx].reset_index(drop=True)
    labels_filtered = labels[cluster_idx]
    
    if print_info == True:
        print("Filtered dataset label distribution:")
        label_counts_after = pd.Series(labels_filtered).value_counts()
        print(label_counts_after)
    
        print(f"Total points before filtering: {df.shape[0]}, after filtering: {df_filtered.shape[0]}")
        print(f"Removed {df.shape[0] - df_filtered.shape[0]} background points.")

    if detected_labels is not None:
        detected_labels_filtered = detected_labels[cluster_idx]
        return df_filtered, labels_filtered, detected_labels_filtered
    
    
    return df_filtered, labels_filtered, None

# Shell grid

In [4]:
def cartesian_to_spherical(points):
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    r = np.sqrt(x ** 2 + y ** 2 + z ** 2)
    theta = np.arccos(np.clip(z / np.maximum(r, 1e-8), -1.0, 1.0))
    phi = np.arctan2(y, x)
    return r, theta, phi

def visualize_shell_grid(df, num_shells, num_sectors):
    fig, ax = plt.subplots(figsize=(10, 10))
    r_max = np.sqrt((df[['x', 'y']] ** 2).sum(axis=1)).max()
    r_bins = np.linspace(0, r_max, num_shells + 1)
    for r in r_bins:
        circle = plt.Circle((0, 0), r, color='black', linestyle='--', fill=False, linewidth=1, alpha=0.5)
        ax.add_patch(circle)

    theta_lines = np.linspace(0, 2 * np.pi, num_sectors + 1)
    for theta in theta_lines:
        ax.plot([0, r_max * np.cos(theta)], [0, r_max * np.sin(theta)], color='black', linestyle='--', linewidth=1, alpha=0.5)

    ax.set_xlim(-r_max, r_max)
    ax.set_ylim(-r_max, r_max)
    ax.set_aspect('equal')
    ax.set_title('Shell Grid')
    plt.show()

def partition_into_shell_grid(df, num_shells, num_sectors, min_points=10):
    points = df[['x', 'y', 'z']].values
    r, theta, phi = cartesian_to_spherical(points)
    phi = np.mod(phi, 2 * np.pi)

    phi_bins = np.linspace(0, 2 * np.pi, num_sectors + 1)
    r_bins = np.linspace(0, r.max(), num_shells + 1)

    phi_indices = np.digitize(phi, bins=phi_bins) - 1
    r_indices = np.digitize(r, bins=r_bins) - 1

    grid_partitions = {}
    for idx, (r_idx, phi_idx) in enumerate(zip(r_indices, phi_indices)):
        cell = (r_idx, phi_idx)
        if cell not in grid_partitions:
            grid_partitions[cell] = []
        grid_partitions[cell].append(idx)

    # Ensure all partitions have at least min_points, merge small ones
    valid_partitions = {k: v for k, v in grid_partitions.items() if len(v) >= min_points}
    if len(valid_partitions) < len(grid_partitions):
        all_indices = [idx for indices in grid_partitions.values() for idx in indices]
        valid_partitions[(0, 0)] = all_indices  # Merge into single valid sector if necessary

    return valid_partitions

In [5]:
def visualize(df, rotation_step, shift_step, projection, true_labels, colors, alphas, zorders):
    fig, ax = plt.subplots(figsize=(10, 10))
    if projection == 'xy':
        points = df[['x', 'y']].values
    else:
        points = df[['x', 'z']].values

    true_labels = true_labels.astype(int)
    unique_labels = np.unique(true_labels)

    for label in unique_labels:
        idx = true_labels == label
        if label == 0:
            color = colors[-1]
            alpha = alphas[-1]
            zorder = zorders[-1]
        else:
            color = colors[(label - 1) % (len(colors) - 1)]
            alpha = alphas[(label - 1) % (len(alphas) - 1)]
            zorder = zorders[(label - 1) % (len(zorders) - 1)]
        ax.scatter(points[idx, 0], points[idx, 1], s=10, color=color, alpha=alpha, zorder=zorder, label=f'true label {label}')

    r_max = np.sqrt((points ** 2).sum(axis=1)).max()
    theta_lines = np.linspace(0, 2 * np.pi, 12 + 1)
    for theta in theta_lines:
        ax.plot([0, r_max * np.cos(theta)], [0, r_max * np.sin(theta)], color='black', linestyle='--', linewidth=1, alpha=0.5)

    r_bins = np.linspace(0, r_max, 6 + 1)
    for r in r_bins:
        circle = plt.Circle((0, 0), r, color='black', linestyle='--', fill=False, linewidth=1, alpha=0.5)
        ax.add_patch(circle)

    ax.set_title(f'Rotation {rotation_step + 1}, Shift {shift_step + 1} with true colors')
    ax.legend()
    plt.show()

def generate_random_colors(n):
    cmap = plt.colormaps.get_cmap("tab20")
    return [cmap(i % (cmap.N - 1)) for i in range(n)]

def visualize_sector(df_sector, true_labels_in_sector, cluster_labels, sector_idx, projection, colors):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    if projection == 'xy':
        points = df_sector[['x', 'y']].values
    else:
        points = df_sector[['x', 'z']].values
    
    unique_true_labels = sorted(np.unique(true_labels_in_sector))  
    unique_cluster_labels = sorted(np.unique(cluster_labels))
    
    # true labels plot
    ax1 = axes[0]
    true_label_colors = {int(label): colors[int(label) - 1] for label in unique_true_labels if label != 0}  
    true_label_colors[0] = colors[-1] # assign the last color for label 0
    
    for label in unique_true_labels:
        idx = true_labels_in_sector == label
        ax1.scatter(points[idx, 0], points[idx, 1], s=20, color=true_label_colors[int(label)], label=f'true label {int(label)}')
    ax1.set_title(f'Sector {sector_idx} - true labels')
    ax1.legend()
    
    #clustering labels plot
    ax2 = axes[1]
    if len(unique_cluster_labels) > 0:
        cluster_colors = generate_random_colors(len(unique_cluster_labels))
        cluster_color_map = {label: cluster_colors[i] for i, label in enumerate(unique_cluster_labels)}
        
        for label in unique_cluster_labels:
            idx = cluster_labels == label
            ax2.scatter(points[idx, 0], points[idx, 1], s=20, color=cluster_color_map[label], label=f'cluster {int(label)}')
        ax2.set_title(f'Sector {sector_idx} - clustered labels')
        ax2.legend()
        
        # detected cluster counts
        cluster_counts = {label: np.sum(cluster_labels == label) for label in unique_cluster_labels}
        print(f"      Detected cluster counts in sector {sector_idx}: {cluster_counts}")
    else:
        ax2.set_title(f'Sector {sector_idx} - no clusters found')
        ax2.axis('off')
    
    plt.show()


In [6]:
def run_shell_grid_consensus_clustering(df, K, true_labels, solver, consensus_nclass, random_state=42,
                                        colors=None, alphas=None, zorders=None, num_rotations=3, rotation_axis='z',
                                        rotation_angle=None, num_shifts=3, num_sectors=3, num_shells=3, projection='xy',
                                        max_iter=1_000, method="random", theta=0.001, epsilon=0.0001, initial_partition=None,
                                        rho_multiplier=None, verbose=False, th_dist=None, dist_max=1e3,
                                        print_info=False, entire_G=False):
    
    start_time = time.time()
    if print_info == True:
        visualize_shell_grid(df, num_shells, num_sectors)

    original_dataset_size = len(df)
    if not isinstance(true_labels, pd.Series):
        true_labels = pd.Series(true_labels)

    if rotation_angle is None:
        rotation_angle = 360 / num_rotations
    
    smc = SemiMetricDensity(data=df, K=K, max_iter=max_iter, method=method)

    # Calculate entire G matrix at begining
    if entire_G == True:
        smc.compute_G(th_dist=th_dist, dist_max=dist_max) 
        G = smc.G 

    grid_results = []
    for rotation_step in range(num_rotations):
        angle = rotation_step * rotation_angle
        if rotation_step == 0:
            rotated_df = df
        else:
            rotated_df = consensus_clusterings.rotate_dataset(df, angle, rotation_axis)
        
        rotated_indices = np.arange(original_dataset_size)

        if print_info == True:
            print(f"Rotation {rotation_step + 1}: Angle {angle} degrees")

        for shift_step in range(num_shifts):
            label_offset = 0
            grid_partitions = partition_into_shell_grid(rotated_df, num_shells, num_sectors)
            shift_results = []

            if print_info == True:
                print(f"  Shift {shift_step + 1}:")

            for cell_idx, (cell, indices) in enumerate(grid_partitions.items()):
                if len(indices) == 0:
                    continue

                original_indices = rotated_indices[indices]
                if entire_G == False:
                    smc.compute_G(original_indices, th_dist, dist_max)
            
                else:
                    smc.G = G[np.ix_(original_indices, original_indices)]
                
                p_i, _  = smc.run_softmax(theta=theta, epsilon=epsilon, initial_partition=initial_partition, rho_multiplier=rho_multiplier, verbose=verbose)

                raw_cluster_labels = np.argmax(p_i, axis=1)
                unique_raw_labels = np.unique(raw_cluster_labels)
                
                label_mapping = {}
                for new_label, old_label in enumerate(unique_raw_labels):
                    label_mapping[old_label] = new_label + label_offset

                cluster_labels = []
                for label in raw_cluster_labels:
                    cluster_labels.append(label_mapping[label])
                cluster_labels = np.array(cluster_labels)

                label_offset += len(unique_raw_labels)

                shift_results.append((cluster_labels, original_indices))

                if print_info == True:
                    print(f"    Sector {cell_idx + 1}: {len(unique_raw_labels)} clusters, Labels: {list(label_mapping.values())}, Points: {len(indices)}")

                if isinstance(true_labels, pd.Series):
                    true_labels_in_sector = true_labels.iloc[original_indices]
                else:
                    true_labels_in_sector = pd.Series(true_labels[original_indices])
                true_label_counts = true_labels_in_sector.value_counts().to_dict()
                
                if print_info == True:
                    print(f"      True label counts in sector {cell_idx + 1}: {true_label_counts}")

                if print_info == True:
                    visualize_sector(rotated_df.iloc[original_indices], true_labels_in_sector, cluster_labels, cell_idx + 1, projection, colors)

            grid_results.append((rotation_step, shift_step, shift_results))
            if print_info == True:
                visualize(rotated_df, rotation_step, shift_step, projection, true_labels, colors, alphas, zorders)

    consensus_results = consensus_clusterings.consensus_clustering_from_grid(grid_results, solver=solver, nclass=consensus_nclass, random_state=random_state, verbose=verbose)

    final_time = time.time() - start_time
    if print_info == True:
        print(f"Needed time: {final_time:.2f}s")
         
    return consensus_results

In [7]:
def experiment_shell_grid_clustering(df, true_labels, 
                                     K_values, num_rotations_values, num_shifts_values, 
                                     num_shells_values, num_sectors_values, rotation_angles, 
                                     solvers, consensus_nclass_values, theta_values, 
                                     epsilon_values, th_dist_values, dist_max_values, 
                                     projection='xy', colors=None, alphas=None, 
                                     zorders=None, output_file='clustering_results.csv'):
    
    results = []
    configurations = []
    
    # Generate all parameter combinations
    param_combinations = list(itertools.product(
        K_values, num_rotations_values, num_shifts_values, num_shells_values, num_sectors_values, 
        rotation_angles, solvers, consensus_nclass_values, theta_values, epsilon_values, 
        th_dist_values, dist_max_values
    ))
    
    # Initialize progress bar
    with tqdm(total=len(param_combinations), desc="Running Experiments") as pbar:
        for params in param_combinations:
            (K, num_rotations, num_shifts, num_shells, num_sectors, rotation_angle, solver, 
             consensus_nclass, theta, epsilon, th_dist, dist_max) = params
            
            start_time = time.time()
            
            consensus_labels = run_shell_grid_consensus_clustering(
                df, K, true_labels, solver, consensus_nclass,
                random_state=42, colors=colors, alphas=alphas, zorders=zorders,
                num_rotations=num_rotations, rotation_axis='z',
                rotation_angle=rotation_angle, num_shifts=num_shifts, 
                num_sectors=num_sectors, num_shells=num_shells,
                projection=projection, max_iter=1000, method="random",
                theta=theta, epsilon=epsilon, initial_partition=None,
                rho_multiplier=None, verbose=False, th_dist=th_dist, 
                dist_max=dist_max, print_info=False, entire_G=False)
            
            runtime = time.time() - start_time
            
            # Compute NMI without background points
            nmi = consensus_clusterings.calculate_nmi(consensus_labels, true_labels)
            
            # Save results
            results.append([K, num_rotations, num_shifts, num_shells, 
                           num_sectors, rotation_angle, solver, 
                           consensus_nclass, theta, epsilon, th_dist, 
                           dist_max, nmi, runtime])
            
            configurations.append({
                "K": K,
                "Num Rotations": num_rotations,
                "Num Shifts": num_shifts,
                "Num Shells": num_shells,
                "Num Sectors": num_sectors,
                "Rotation Angle": rotation_angle,
                "Solver": solver,
                "Consensus NClass": consensus_nclass,
                "Theta": theta,
                "Epsilon": epsilon,
                "Th_dist": th_dist,
                "Dist_max": dist_max,
                "NMI": nmi,
                "Runtime": runtime
            })
            
            # Update progress bar
            pbar.update(1)
    
    results_df = pd.DataFrame(results, columns=['K', 'Num Rotations', 'Num Shifts', 'Num Shells', 
                                                'Num Sectors', 'Rotation Angle', 'Solver', 
                                                'Consensus NClass', 'Theta', 'Epsilon', 'Th_dist', 
                                                'Dist_max', 'NMI', 'Runtime'])
    
    results_df.to_csv(output_file, index=False)
    
    print(f"Results saved to {output_file}")
    return results_df


In [8]:
# The best and the worst options
def get_best_and_worst_options(results_file, top_n=10):
   
    results_df = pd.read_csv(results_file)
    
    # Sort dataframe by NMI
    best_options = results_df.nlargest(top_n, 'NMI')
    worst_options = results_df.nsmallest(top_n, 'NMI')
    
    return best_options, worst_options

In [9]:
def rerun_shell_grid_clustering_on_detected_cluster(df, true_labels, detected_cluster_id, consensus_labels, 
                                                    K, solver, consensus_nclass, num_rotations, num_shifts, 
                                                    num_sectors, num_shells, projection='xy', colors=None, 
                                                    alphas=None, zorders=None, rotation_axis='z', 
                                                    rotation_angle=None, max_iter=1000, method="random", 
                                                    theta=0.001, epsilon=0.0001, th_dist=None, dist_max=1000, 
                                                    print_info=False, entire_G=False):
    
    # Select points belonging only to the specified detected cluster
    cluster_indices = np.where(consensus_labels == detected_cluster_id)[0]
    subset_df = df.iloc[cluster_indices].reset_index(drop=True)
    
    if isinstance(true_labels, pd.Series):
        subset_true_labels = true_labels.iloc[cluster_indices].reset_index(drop=True)
    else:
        subset_true_labels = true_labels[cluster_indices]

    if len(subset_df) < 2:
        print(f"Cluster {detected_cluster_id}: Not enough points to rerun clustering.")
        return None, None, None, None
    
    if print_info == True:
        print(f"Rerunning clustering on detected cluster {detected_cluster_id} with {len(subset_df)} points.")

    # Run shell grid consensus clustering on the subset
    start_time = time.time()
    clustering_results = run_shell_grid_consensus_clustering(
        subset_df, K, subset_true_labels, solver, consensus_nclass,
        random_state=42, colors=colors, alphas=alphas, zorders=zorders,
        num_rotations=num_rotations, rotation_axis=rotation_axis, rotation_angle=rotation_angle,
        num_shifts=num_shifts, num_sectors=num_sectors, num_shells=num_shells,
        projection=projection, max_iter=max_iter, method=method,
        theta=theta, epsilon=epsilon, initial_partition=None,
        rho_multiplier=None, verbose=False, th_dist=th_dist, 
        dist_max=dist_max, print_info=print_info, entire_G=entire_G
    )
    clustering_time = time.time() - start_time

    subset_true_labels = subset_true_labels[:len(clustering_results)]

    return clustering_results, subset_df, subset_true_labels, clustering_time